# KV Cache 池化技术概述

随着大模型规模和上下文长度的增长，单节点显存已无法满足 KV Cache 的存储需求。本节将介绍 KV Cache 池化技术的动机、原理和存储介质，帮助你理解从单节点管理到分布式池化的技术演进。

本节学习大纲如下：

- 单节点 KV Cache 管理的局限
- KV Cache 池化的原理
- 池化的存储介质
- 池化技术与 PD 分离的关系
- 应用场景


## 1. 单节点 KV Cache 管理的局限

### 1.1 显存容量瓶颈

KV Cache 占用大量 NPU 显存，其大小与模型规模、并发请求数和序列长度直接相关：

```
KV Cache 大小 = num_layers × 2 × batch_size × seq_len × num_heads × head_dim × dtype_size
```

公式中每一项的含义：

| 参数 | 含义 | 说明 |
| --- | --- | --- |
| `num_layers` | Transformer 层数 | 每层有独立的 KV Cache，层间不共享 |
| `2` | K 和 V | 每个 token 产生 Key（算注意力分数）和 Value（加权求和）两个向量 |
| `batch_size` | 并发请求数 | 每个请求有各自的 KV Cache |
| `seq_len` | 序列长度 | 每个 token 位置都要存一对 K/V，**生成过程中不断增长** |
| `num_heads` | 注意力头数 | 多头注意力，每个头都有独立的 K/V，捕捉不同语义关系 |
| `head_dim` | 每个头的维度 | K/V 向量的长度，决定单个 token 的信息容量 |
| `dtype_size` | 每个元素字节数 | FP16=2, FP32=4。**降精度是最直接的省内存手段** |

下面通过代码计算不同模型配置下的 KV Cache 大小：



In [ ]:
# 计算不同模型配置下的 KV Cache 大小
def calc_kv_cache(num_layers, batch_size, seq_len, num_heads, head_dim, dtype_size=2):
    """计算 KV Cache 大小（GB）"""
    return num_layers * 2 * batch_size * seq_len * num_heads * head_dim * dtype_size / (1024**3)

# 常见模型配置: (名称, 层数, batch, seq_len, 头数, 头维度)
models = [
    ("LLaMA-7B",  32, 32, 4096, 32, 128),
    ("LLaMA-13B", 40, 32, 4096, 40, 128),
    ("LLaMA-65B", 80, 32, 4096, 64, 128),
]

print(f"{'模型':<15} {'batch':>6} {'seq_len':>8} {'KV Cache (FP16)':>16} {'单卡64GB够？':>12}")
print("-" * 65)
for name, layers, batch, seq, heads, dim in models:
    kv = calc_kv_cache(layers, batch, seq, heads, dim)
    fits = "够" if kv < 64 else "不够"
    print(f"{name:<15} {batch:>6} {seq:>8} {kv:>13.1f} GB {fits:>12}")

# 试试：把 batch_size 改成 1，seq_len 改成 32768，看看会怎样？


### 1.2 KV Cache 是一次性的，无法复用

没有池化时，KV Cache 的生命周期是：计算 → 传输给 Decode → 用完即弃。如果同一 prompt 被多次请求（如多轮对话、相同问题批量推理），每次都需要重新计算 Prefill，无法复用已计算过的 KV Cache，造成算力浪费。

### 1.3 KV Cache 无法跨节点共享

PD 分离场景中，Prefill 产出的 KV Cache 传输到 Decode 后即被消费，不同节点之间没有共享存储。空闲节点的内存资源无法被高负载节点利用，造成资源利用率不均衡。


## 2. KV Cache 池化的原理

### 2.1 核心思想

KV Cache 池化将多个节点的内存资源汇聚成统一的 KV Cache 池，KV Cache 可跨节点存取，实现资源共享。

<img src="./images/kv_cache_pooling_concept.svg" alt="KV Cache 池化概念图" width="80%">

### 2.2 工作流程

- Prefill 节点计算完 KV Cache 后，从 NPU 显存写入分布式池（D2H：Device → Host DRAM）
- Decode 节点从池中读取所需 KV Cache 到 NPU 显存（H2D：Host DRAM → Device）
- 后续相同请求可从池中直接复用 KV Cache，跳过 Prefill 计算

### 2.3 池化的价值

| 价值 | 说明 |
| --- | --- |
| 突破单节点显存上限 | 利用多节点 Host DRAM 扩展容量 |
| KV Cache 复用 | 相同 prompt 的 KV Cache 只算一次 |
| 负载均衡 | 空闲节点内存可被高负载节点利用 |
| 弹性扩展 | 池容量随节点增加线性扩展 |


## 3. 池化的存储介质

### 3.1 为什么不用 NPU 显存做池？

<img src="./images/storage_hierarchy.svg" alt="存储介质层次" width="95%">

主要有两个原因：

**容量不够**：NPU 显存被模型权重和当前推理的 KV Cache 占用，剩余空间不足以做共享池。以 64GB 显存为例，模型权重约占 30GB，当前请求的 KV Cache 约占 20GB，剩余仅 14GB，远不够存储多个请求的历史 KV Cache。

**无法跨节点访问**：NPU 显存是每张卡私有的，机器 A 的 NPU 显存无法被机器 B 直接寻址读写。而 Host DRAM 可以通过 RDMA 网络跨机器访问（HCCS 用于机内），这是实现跨节点共享池的前提。

### 3.2 Host DRAM 内存池

池化系统将每个节点贡献的 Host 内存汇聚成全局内存池，由一个中心协调者统一管理空间分配。Host DRAM 可通过 RDMA/HCCS 跨节点访问，且容量远大于单卡 HBM。

### 3.3 SSD offload（可选）

当 Host DRAM 容量不足时，池化系统支持将不活跃的 KV Cache 溢出到 SSD，进一步扩展池容量。SSD 访问延迟高于 DRAM，适合存放冷数据。

> **注意**：NPU HBM 中的 KV Cache 由推理引擎（vLLM/SGLang）自行管理，不属于池化系统的范畴。池化系统管理的是 Host DRAM 内存池和可选的 SSD 存储。


## 4. 池化技术与 PD 分离的关系

第五章介绍了 PD 分离传输（llm-datadist / Transfer Engine 直传），本章介绍 KV Cache 池化（Store put/get）。两者解决不同层面的问题，**不是二选一，而是可以组合使用**。

<img src="./images/pd_vs_pooling_comparison.svg" alt="PD 分离 vs 池化数据路径对比" width="90%">

### 4.1 两种能力的定位

| 维度 | Transfer Engine 直传 | Store 池化（put/get） |
| --- | --- | --- |
| 定位 | **传输**：实时搬运 KV Cache | **存储**：缓存复用 KV Cache |
| 数据路径 | D2D 直传（不经 Host） | D2H → Host DRAM 池 → H2D |
| 数据生命周期 | 即传即用，用完即弃 | 可持久化缓存，跨请求复用 |
| 寻址方式 | 点对点（指定目标地址） | key-value（按 key 存取） |
| 适用场景 | PD 分离的实时 KV 迁移 | 前缀缓存、跨实例共享、容量扩展 |

### 4.2 组合使用

实际部署中两者可以同时启用：

- **PD 实时传输**用 TE 直传：Prefill NPU → D2D → Decode NPU，低延迟
- **缓存复用**用 Store 池化：Prefill → D2H → Store 存一份 → 下次命中直接取，省算力

例如 vLLM 通过 `MultiConnector` 同时挂载 `MooncakeConnector`（TE 直传）和 `MooncakeStoreConnector`（Store 池化），实现 PD 实时迁移 + 跨实例前缀缓存。SGLang 也可以同时开启 PD 分离（TE 直传）和 HiCache（Store 作为 L3 后端）。

> **下一节预告**：Mooncake Store 就是一个实现了上述池化能力的分布式系统。下一节将介绍它的架构设计——Master（协调者）、Store（存储层）、Transfer Engine（传输引擎）如何协作完成 KV Cache 的跨节点存储与共享。


## 5. 应用场景

### 5.1 多轮对话

用户与 AI 助手多轮对话，每轮都会带上之前所有对话内容：

```
第 1 轮：用户问 "什么是昇腾"
  → Prefill 计算 5 个 token 的 KV Cache
  → 存入池

第 2 轮：用户追问 "和英伟达比呢"
  完整 prompt = "什么是昇腾，和英伟达比呢"
  → 前缀 "什么是昇腾" 命中池缓存，直接取回（省掉重算）
  → 只需 Prefill 新增部分 "，和英伟达比呢"

第 3 轮：用户再问 "支持哪些模型"
  完整 prompt = "什么是昇腾，和英伟达比呢，支持哪些模型"
  → 前缀命中池缓存，只算新增部分
```

对话越长，节省的算力越多。没有池化时，每一轮都要从头重算所有历史对话的 KV Cache。

### 5.2 相同 prompt 批量推理

多个用户发送相同或相似的 prompt（如客服场景的常见问题）：

```
用户 A："公司报销流程是什么？"     → Prefill 计算，存入池
用户 B："公司报销流程是什么？"     → 命中池缓存，跳过 Prefill
用户 C："公司报销流程是什么？"     → 命中池缓存，跳过 Prefill
用户 D："公司报销流程怎么查看？"   → 前缀命中，只算 "怎么查看"
```

高频重复问题只算一次，后续请求直接从池中读取，大幅降低 TTFT（首 Token 响应时间）。

### 5.3 PD 分离中的跨实例缓存共享

PD 分离架构中通常有多个 Prefill 实例和多个 Decode 实例。TE 直传负责当前请求的实时 KV 迁移（Prefill → Decode），Store 则负责跨实例的 KV Cache 共享：

```
Prefill 实例 1 算完 "什么是昇腾" → Store put（缓存）

Prefill 实例 2 收到 "什么是昇腾，怎么安装"
  → 查 Store：前缀 "什么是昇腾" 命中
  → Store get 取回，跳过这部分 Prefill 计算
  → 只算新增的 "，怎么安装"
  → TE 直传给 Decode 实例
```

TE 直传管"当前请求的实时传输"，Store 管"跨实例、跨请求的缓存复用"，两者各司其职。

### 5.4 长上下文场景

超长上下文（如代码分析、长文档摘要）的 KV Cache 可能超出单卡显存：

```
128K 上下文 × FP16 → 单层 KV Cache 约 1GB
32 层 → 总计约 32GB → 超出单卡 64GB 显存（还要放模型权重）

解决方案：
  活跃的部分留在 NPU HBM（快速访问）
  不活跃的部分卸载到池中（Host DRAM，按需 H2D 取回）
```

池化扩展了有效显存容量，让长上下文推理不再受限于单卡 HBM。


## 课后练习

本节介绍了 KV Cache 池化技术的动机、原理、存储介质以及与 PD 分离的关系，请根据学习内容完成以下题目进行自测。

1. （判断题）KV Cache 池化技术可以将多个节点的内存资源汇聚成统一的缓存池，实现跨节点共享。

2. （单选题）KV Cache 池化主要基于哪种介质构建？
    A. NPU HBM
    B. Host DRAM
    C. SSD
    D. 网络缓冲区

3. （单选题）以下哪项不是 KV Cache 池化的核心价值？
    A. 突破单节点显存上限
    B. KV Cache 复用
    C. 降低模型参数量
    D. 弹性扩展

4. （单选题）KV Cache 池化与 PD 分离传输的主要区别是什么？
    A. 池化使用点对点传输，PD 分离使用发布-订阅
    B. 池化的数据路径是 D2H/H2D，PD 分离是 D2D 直传
    C. 池化不支持 KV Cache 复用
    D. PD 分离传输的数据生命周期更长

5. （多选题）KV Cache 池化适用于哪些场景？
    A. 多轮对话
    B. 相同 prompt 批量推理
    C. PD 分离中的跨实例缓存共享
    D. 长上下文场景

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/06.02_answer.txt